# Identificación de efectos causales

La buena econometría requiere buen uso de la teoría económica y solidez conceptual. Los siguientes ejercicios son para que analice y argumente

## Informalidad e ingresos

¿Cuál es el efecto de la informalidad laboral sobre la remuneración al trabajo? Dicho de otra manera, ¿ser trabajajador informal implica tener ingresos más bajos?

### Datos

Trabajaremos con la base de datos Personas de la medición de pobreza monetaria para 2025, para el dominio Barranquilla AM. [Acá](https://microdatos.dane.gov.co/index.php/catalog/908/data-dictionary/F7?file_name=Personas) puede consultar el diccionario de variables

In [ ]:
suppressWarnings(
pacman::p_load(xtable,dplyr, sandwich, lmtest)
)

In [ ]:
file<-"https://raw.githubusercontent.com/andvarga-eco/econometrics_ml/refs/heads/main/data/Personas2025_baq.csv"
data<-read.csv(file)

### Definición de variables

Trabajaremos con el ingreso por hora en logarítmo. Como medida de ingreso laboral usaremos la suma del ingreso monetario de la primera actividad y de la segunda actividad. En ambos casos, antes de imputación. Filtrar para personas ocupadas. Definimos un trabajador informal como aquel que no cotiza a un fondo de pensiones, P6920

In [ ]:
#Salario por hora
data<-data|>filter(oc==1)|>
  mutate(w=(impa+isa)/(p6800*4))|>filter(w>0)|>mutate(lw=log(w))

In [ ]:
# Informal
data<-data|>filter(p6920!=3)|>mutate(inf=ifelse(p6920==1,0,1))

In [ ]:
# Años de educación

data<- data|>
  mutate(
    # Piso de años acumulados según el nivel educativo alcanzado
    piso = case_when(
      p3042 == 1 ~ 0,
      p3042 == 2 ~ 0,
      p3042 == 3 ~ 0,
      p3042 == 4 ~ 5,
      p3042 %in% c(5, 6, 7) ~ 9,
      p3042 %in% c(8, 9, 10) ~ 11,
      p3042 %in% c(11, 12, 13) ~ 17,
      TRUE ~ NA_real_
    ),

    # Grado/semestre reportado, limpiando no informa (99) y missing
    grado_raw = ifelse(p3042s1 %in% c(99, NA), 0, p3042s1),

    # Para niveles >= 8 (técnica profesional en adelante), p6210s1 son SEMESTRES:
    # se convierten a años completos truncando (semestre impar -> año anterior)
    # Ej: 1 sem -> 0 años | 2 sem -> 1 año | 3 sem -> 1 año | 4 sem -> 2 años
    grado = case_when(
      p3042 %in% c(8, 9, 10, 11, 12, 13) ~ floor(grado_raw / 2),
      TRUE ~ grado_raw
    ),

    anios_educ = piso + grado,
    anios_educ = ifelse(p3042 == 1, 0, anios_educ)
  ) %>%
  select(-grado_raw)

In [ ]:
# Mujer
data<-data|>mutate(mujer=ifelse(p3271==2,1,0))

In [ ]:
# Experiencia
data<-data|>mutate(exp=p6040-anios_educ-6,exp2=exp^2)

## Estimación

In [ ]:
options(warn = -1)
basic<-lw~inf
regbasic<-lm(basic,data)
cat("Coeficiente y error estándar robusto de Inf",coeftest(regbasic, vcov = vcovHC(regbasic, type = "HC1"))[2,c(1:2)], "\n")

### Interpretación

Use la descomposición y argumente por qué el valor estimado es o no es un efecto causal

\begin{aligned}
\underbrace{E[Y \mid D=1] - E[Y \mid D=0]}_{\text{SDO}}
  &= \underbrace{E[Y(1)] - E[Y(0)]}_{\text{ATE}} \\
  &\quad + \underbrace{E[Y(0) \mid D=1] - E[Y(0) \mid D=0]}_{\text{Sesgo de selección}} \\
  &\quad + \underbrace{(1-\pi)(ATT - ATU)}_{\text{Heterogeneous treatment effect bias}}
\end{aligned}

### Estimación con covariables

In [ ]:
options(warn = -1)
basic_cov<-lw~inf+mujer+anios_educ+exp+exp2+C(p6430)
regbasic_cov<-lm(basic,data)
cat("Coeficiente y error estándar robusto de Inf",coeftest(regbasic_cov, vcov = vcovHC(regbasic_cov, type = "HC1"))[2,c(1:2)], "\n")

¿Puede interpretar el coeficiente de informalidad como CATE? Explique detalladamente su razonamiento